# 01 · Comprensión del negocio

| | |
|---|---|
| **Fase CRISP-DM** | 1 — Comprensión del negocio |
| **Objetivo** | Definir el problema, los objetivos de negocio y de minería de datos, los criterios de éxito y el alcance de la PoC VoltCast. |
| **Entradas** | `docs/historias_de_usuario_voltcast.md`, `docs/detalle-propuestas-modelos-ml.md` |
| **Salidas** | Ninguna (documento de referencia para los notebooks siguientes) |

## 1. Contexto

La planta opera dos **electrolizadores, EL A y EL B**, cada uno formado por **181 celdas conectadas en serie**. Por cada electrolizador circula la misma corriente (≈ 15 kA), y el voltaje total (`[Volts]`) es la suma de los voltajes de las celdas más las caídas en barras y conexiones.

El voltaje de cada celda es el principal indicador de su estado:

- **Consumo energético.** La energía que consume un electrolizador es proporcional a `V × I`. Con 362 celdas, un aumento de 0,1 V por celda representa una parte relevante de la factura eléctrica.
- **Degradación.** Con el envejecimiento de membranas y electrodos, el voltaje de la celda sube lentamente durante meses. Anticipar esa subida permite planificar reemplazos.
- **Mantenimiento.** Una celda que se acerca al voltaje máximo permitido debe revisarse o puentearse. Conocerlo con 2 a 6 meses de antelación permite programar la parada, comprar repuestos y evitar paros no planificados.

Hoy el seguimiento es **reactivo**: se mira el voltaje del día. VoltCast busca pasar a un seguimiento **predictivo**, con el pronóstico del voltaje de cada celda en una fecha futura.

## 2. Objetivos de negocio

Los objetivos se vinculan con las historias de usuario del módulo de *Machine Learning* (ver `docs/historias_de_usuario_voltcast.md`):

| Historia | Necesidad del Ingeniero de Planta | Cómo la cubre la PoC |
|---|---|---|
| **US005** Generación de proyecciones | Generar proyecciones de voltaje con un modelo entrenado con los datos de la planta. | Función `predecir(fecha, celda)` y pronóstico completo a 180 días (notebook 08). Versionado del modelo para el caso de reentrenamiento (E02). |
| **US006** Precisión del modelo | Conocer la efectividad del sistema (MSE, RMSE, MAE, MAPE). | Evaluación *rolling-origin* en V por celda contra la señal real, con comparación contra baselines (notebook 07). |
| **US007** Visualización de pronósticos | Identificar celdas cuyo pronóstico supere el voltaje permitido. | Exportación de `pronosticos.parquet` con todas las celdas, lista para visualizar y resaltar excesos. |
| **US008** Filtrado por fecha | Consultar los pronósticos de una fecha concreta. | `pronosticos.parquet` indexado por `fecha_objetivo`, filtrable por fecha. |

Las historias **US001–US004** (carga, identificación de columnas y celdas, limpieza) se cubren con las validaciones de la ingesta Bronze (notebook 02) y las reglas de Silver (notebook 04).

## 3. Objetivo de minería de datos

> **Pronosticar el voltaje de cada celda individual** (2 × 181 = 362 series) para **cualquier día entre 1 y 180 días** después de la última fecha observada, con énfasis en los horizontes de **60 y 180 días**.

- **Tipo de problema:** regresión de series temporales, multi-horizonte y multi-serie.
- **Granularidad:** diaria, por celda.
- **Arquitectura:** **BO-VMD + Bi-NOA-LSTM (multi-horizonte directo)**, la Propuesta A simplificada.
  - **BO-VMD:** Descomposición Variacional en Modos (VMD) con hiperparámetros `K` y `alpha` elegidos por optimización bayesiana, para separar el ruido de alta frecuencia de la tendencia.
  - **Bi-NOA-LSTM:** LSTM bidireccional sin activación en la salida del estado oculto (`h_t = o_t ⊙ c_t`).
  - **Salida directa:** la red emite los 180 días en un único pase, sin recursión.
- **Entrada:** ventana de 30 días (`LOOKBACK_DAYS`) de voltaje filtrado de la celda, corriente (`kA`) y voltaje medio del electrolizador.

## 4. Criterios de éxito

La PoC se considera **exitosa** si se cumplen los tres criterios:

1. **Supera al mejor baseline** en RMSE a **60 y 180 días**. Los baselines son la *persistencia* (último nivel observado) y la *tendencia lineal* ajustada a la ventana de entrada.
2. **Curva de error vs. horizonte** (día 1 → 180), reportada y graficada, sin deriva por recursión. La inferencia es directa en un solo pase.
3. **Métricas en voltios (V) por celda**, calculadas contra la **señal real sin filtrar**, solo en fechas con calidad válida.

> La inferencia directa elimina la acumulación de error *recursiva*, pero la incertidumbre crece naturalmente con el horizonte. Eso se mide; no se asume que sea nulo.

## 5. Supuestos, restricciones y riesgos

**Supuestos**
- El registro diario es representativo del estado de la celda ese día.
- La degradación es lenta y continua, salvo intervenciones (reemplazos, puenteos) visibles en los datos.
- La corriente (`kA`) y el voltaje medio del electrolizador son covariables comunes que resumen la operación.

**Restricciones**
- Un único archivo de origen: `VoltajesDiariosDesde2020.xlsx`, con datos diarios del 2020-01-01 al 2026-08-27 (~2420 filas por hoja).
- Hardware local: una GPU RTX 4060 (8 GB). Los notebooks también deben poder ejecutarse en Google Colab.
- Los datos son confidenciales: `data/` no se sube al repositorio.

**Riesgos**

| Riesgo | Mitigación |
|---|---|
| Pocos datos: ~1935 días de train y ~484 de validación; los ~305 orígenes posibles para H = 180 están muy solapados. | Pesos compartidos entre las 362 celdas (canal independiente), dropout y *early stopping*. |
| El esquema de canal independiente no modela explícitamente la relación entre celdas. | Covariables comunes del electrolizador (`kA`, `V_total`). |
| Los parámetros de VMD comunes suponen un ruido parecido entre celdas. | Verificación de homogeneidad en el notebook 05. |
| Costo de la VMD causal en validación (~25 000 descomposiciones). | `EVAL_STRIDE`, `VMD_HISTORY` y paralelización con `joblib`. |
| Celdas puenteadas o fuera de servicio durante períodos largos. | Máscara de calidad en Silver; esas muestras se excluyen del entrenamiento y de las métricas. |
| Fuga de información del futuro. | Split temporal 80/20; todo ajuste (escaladores, BO-VMD) solo sobre train; VMD causal en validación. |

## 6. Alcance

**Dentro de la PoC**
- Ingesta, limpieza y arquitectura Medallion (Bronze → Silver → Gold).
- BO-VMD para el filtrado, Bi-NOA-LSTM multi-horizonte directo y ablación con Bi-LSTM estándar.
- Evaluación contra baselines y función de pronóstico para despliegue.

**Fuera de la PoC**
- **RFR-SHAP** (selección de covariables) y **SOMA** (optimización de hiperparámetros) de la Propuesta A. Las covariables se fijan por conocimiento del proceso y los hiperparámetros se ajustan de forma manual o con `optuna`.
- Interfaz de usuario, autenticación y modo sin conexión (US007-E03/E04, US008-E02/E03). La PoC entrega los datos que esas funciones consumirían.
- Registro de celdas (US009).

## 7. Plan de trabajo (CRISP-DM × Medallion)

| Notebook | Fase CRISP-DM | Capa | Salida principal |
|---|---|---|---|
| 01 Comprensión del negocio | 1 | — | Este documento |
| 02 Ingesta Bronze | 2 | Bronze | `data/bronze/voltajes_EL_{A,B}.parquet` |
| 03 Comprensión de los datos | 2 | (lee Bronze) | `reports/eda/`, reglas de calidad |
| 04 Preparación Silver | 3 | Silver | `voltajes_silver.parquet`, `split.json` |
| 05 Preparación Gold (BO-VMD) | 3 | Gold | series filtradas, escaladores, orígenes |
| 06 Modelado | 4 | — | `models/bi_noa_lstm.pt` |
| 07 Evaluación | 5 | — | `reports/metricas.csv` |
| 08 Despliegue | 6 | Gold | `pronosticos.parquet`, `predecir()` |

Cada notebook lee solo de la capa anterior y escribe solo en la suya; la lógica compartida vive en `src/voltcast/`.

In [ ]:
# Configuración: localiza la raíz del proyecto (local o Colab) y agrega src/ al path
import sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/Poc-TP1")
except ImportError:
    ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "voltcast").is_dir())
sys.path.insert(0, str(ROOT / "src"))

In [ ]:
from voltcast import config

print(f"Raíz del proyecto : {config.BASE_DIR}")
print(f"Archivo de origen : {config.raw_path()}  (existe: {config.raw_path().exists()})")
print(f"Ventana de entrada: {config.LOOKBACK_DAYS} días | Horizonte: {config.HORIZON_DAYS} días | Cortes: {config.EVAL_HORIZONS}")
print(f"Validación        : {config.VAL_FRACTION:.0%} final de la serie")